# Redrob Candidate Ranker — Sandbox Demo

This notebook runs the full ranking pipeline on 100K candidates and produces a top-100 submission CSV.

**Runtime**: ~2-3 min total (clone + deps + ranking). The ranking step itself takes <30s on CPU.

**Requirements**: Colab runtime with CPU (no GPU needed).

## Step 1: Clone Repository & Install Dependencies

In [ ]:
# Install Git LFS and clone the repository
!git lfs install
!git clone https://github.com/Prateekkp/candidate_ranking.git
%cd candidate_ranking

In [ ]:
# Install Python dependencies
!pip install -q -r requirements.txt

# Verify key packages are installed
import faiss, numpy, pandas
print(f'faiss-cpu: {faiss.__version__}')
print(f'numpy: {numpy.__version__}')
print(f'pandas: {pandas.__version__}')

## Step 2: Verify Pre-computed Artifacts

In [ ]:
from pathlib import Path

artifacts = {
    'candidates.parquet': Path('data/processed/candidates.parquet'),
    'candidates_with_profiles.parquet': Path('data/processed/candidates_with_profiles.parquet'),
    'candidate_index.faiss': Path('data/processed/candidate_index.faiss'),
    'jd_embedding.npy': Path('data/processed/jd_embedding.npy'),
}

for name, path in artifacts.items():
    exists = path.exists()
    size_mb = path.stat().st_size / 1048576 if exists else 0
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}] {name:45s} {size_mb:6.1f} MB')

assert all(p.exists() for p in artifacts.values()), 'Some artifacts are missing! Check Git LFS.'
print('\nAll artifacts present.')

## Step 3: Run Ranking Pipeline

Runs `rank.py` on all 100K candidates. Completes in <30 seconds on CPU.

In [ ]:
!python rank.py --candidates ./data/processed/candidates.parquet --out ./submission.csv

## Step 4: Validate Submission Format

In [ ]:
!python data/raw/validate_submission.py submission.csv

## Step 5: Display Results

In [ ]:
import pandas as pd

df = pd.read_csv('submission.csv')
print(f'Rows: {len(df)}')
print(f'Ranks: {df["rank"].min()} to {df["rank"].max()}')
print(f'Score range: {df["score"].max():.4f} to {df["score"].min():.4f}')
print(f'Unique candidates: {df["candidate_id"].nunique()}')
print()
display(df.head(20))

## Step 6: Download Submission CSV

In [ ]:
from google.colab import files
files.download('submission.csv')